# 09 - Evaluation and Validation Protocol

## Objective

Mengevaluasi candidate generation dan comparison vector dengan label manual pada review sample yang dapat diaudit.

## Problem

Dataset utama tidak memiliki label pasangan bawaan. Label manual hanya berlaku pada review sample, sehingga metrik tidak otomatis mewakili seluruh candidate set atau dataset.

## Evaluation strategy

1. Verifikasi artifact dan konsistensi candidate pair.
2. Ukur coverage blocking terhadap reference deterministic, dengan nama yang tepat: `reference coverage`, bukan recall.
3. Analisis distribusi agreement dan evidence band.
4. Buat sample manual review yang reproducible dan terstratifikasi.
5. Hitung metrik rule secara eksplisit setelah label review tersedia.

## Non-goals

- Tidak menggunakan `customer_id` sebagai ground truth.
- Tidak membuat label fiktif.
- Tidak melakukan automatic merge.
- Tidak menyebut agreement tinggi sebagai precision.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_CANDIDATES = [
    Path.cwd() / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
    Path.cwd().parent / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
]
DATA_PATH = next((path.resolve() for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Dataset terstandardisasi tidak ditemukan. Jalankan 04_standardization.ipynb terlebih dahulu.')

PROCESSED_DIR = DATA_PATH.parent
COMPARISON_PATH = PROCESSED_DIR / 'probabilistic_comparison_vectors.csv'
COVERAGE_PATH = PROCESSED_DIR / 'blocking_reference_coverage.csv'
SUMMARY_PATH = PROCESSED_DIR / 'blocking_candidate_summary.csv'
REVIEW_QUEUE_PATH = PROCESSED_DIR / 'manual_review_queue.csv'

required_paths = [COMPARISON_PATH, COVERAGE_PATH, SUMMARY_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f'Artifact evaluasi belum tersedia: {missing_paths}')

df = pd.read_csv(DATA_PATH).reset_index(names='row_index')
comparison = pd.read_csv(COMPARISON_PATH)
coverage_summary = pd.read_csv(COVERAGE_PATH)
blocking_summary = pd.read_csv(SUMMARY_PATH)

print('Comparison vectors:', comparison.shape)
print('Blocking coverage:', coverage_summary.shape)
print('Blocking summary:', blocking_summary.shape)

## Experiment 1 - Artifact integrity checks

Check ini memastikan pair tidak self-match, row index valid, tidak ada duplicate pair setelah deduplication, dan agreement count konsisten dengan indicator fields.

In [ ]:
agreement_columns = [
    column for column in [
        'email_agree', 'phone_agree', 'name_agree',
        'address_agree', 'city_agree', 'dob_agree',
    ]
    if column in comparison.columns
]
required_comparison_columns = [
    'left_row_index', 'right_row_index', 'agreement_count',
    'available_field_count',
    *agreement_columns,
]
missing_comparison_columns = sorted(set(required_comparison_columns) - set(comparison.columns))
if missing_comparison_columns:
    raise ValueError(f'Kolom comparison vector belum tersedia: {missing_comparison_columns}')

if 'evidence_band' not in comparison.columns:
    def evidence_band(agreement_count: int) -> str:
        if agreement_count >= 4:
            return 'high_agreement_review'
        if agreement_count >= 2:
            return 'medium_agreement_review'
        return 'low_agreement_review'

    comparison['evidence_band'] = comparison['agreement_count'].map(evidence_band)

integrity_checks = {
    'no_self_pairs': bool((comparison['left_row_index'] != comparison['right_row_index']).all()),
    'left_index_valid': bool(comparison['left_row_index'].ge(0).all()),
    'right_index_valid': bool(comparison['right_row_index'].ge(0).all()),
    'unique_pair_rows': bool(~comparison.duplicated(['left_row_index', 'right_row_index']).any()),
    'agreement_count_consistent': bool(
        comparison['agreement_count'].eq(comparison[agreement_columns].sum(axis=1)).all()
    ),
    'agreement_values_binary': bool(
        comparison[agreement_columns].isin([0, 1]).all().all()
    ),
}
integrity_summary = pd.DataFrame({
    'check': list(integrity_checks),
    'passed': list(integrity_checks.values()),
})
integrity_summary

## Experiment 2 - Blocking and evidence diagnostics

Reference coverage bukan recall karena reference set R1-R4 bukan ground truth. Diagnostik ini membantu melihat trade-off jumlah candidate, reduction, dan overlap reference.

In [ ]:
coverage_diagnostics = coverage_summary.merge(
    blocking_summary[['blocking_strategy', 'candidate_reduction_percentage', 'runtime_seconds', 'peak_memory_mb']],
    on='blocking_strategy',
    how='left',
)
coverage_diagnostics

In [ ]:
evidence_diagnostics = (
    comparison.groupby('evidence_band', dropna=False)
    .agg(
        pair_count=('agreement_count', 'size'),
        mean_agreement_count=('agreement_count', 'mean'),
        median_agreement_count=('agreement_count', 'median'),
        max_available_field_count=('available_field_count', 'max'),
    )
    .reset_index()
)
evidence_diagnostics

## Experiment 3 - Stratified manual-review sample

Jika queue manual review sudah tersedia, label yang sudah diisi akan dipertahankan. Jika belum tersedia, sample diambil per `evidence_band` dan provenance blocking dengan `random_state` yang tercatat.

In [ ]:
SAMPLE_PER_GROUP = 25
RANDOM_STATE = 42

queue_loaded_from_disk = REVIEW_QUEUE_PATH.exists()
if queue_loaded_from_disk:
    review_queue = pd.read_csv(REVIEW_QUEUE_PATH, sep=';')
    review_queue['review_label'] = pd.to_numeric(review_queue['review_label'], errors='coerce')
    review_queue['review_status'] = review_queue['review_status'].where(
        review_queue['review_label'].isna(),
        'reviewed',
    )
else:
    review_parts = []
    for (evidence_band_value, strategy_value), group in comparison.groupby(
        ['evidence_band', 'supporting_blocking_strategies'],
        dropna=False,
        sort=True,
    ):
        sample = group.sample(
            n=min(SAMPLE_PER_GROUP, len(group)),
            random_state=RANDOM_STATE,
        ).copy()
        sample['evidence_band'] = evidence_band_value
        sample['supporting_blocking_strategies'] = strategy_value
        review_parts.append(sample)

    review_queue = pd.concat(review_parts, ignore_index=True)
    review_queue['review_status'] = 'pending'
    review_queue['review_label'] = pd.NA
    review_queue['review_notes'] = pd.NA

review_summary = (
    review_queue.groupby(['evidence_band', 'supporting_blocking_strategies'], dropna=False)
    .size()
    .rename('sample_count')
    .reset_index()
)
print('Manual review queue rows:', len(review_queue))
review_summary

## Experiment 3b - Build human-review detail view

`manual_review_queue.csv` hanya berisi row index dan agreement indicator. Gunakan detail view ini untuk melihat dua record berdampingan sebelum mengisi `review_label`.

- Isi `review_label = 1` jika beberapa atribut identitas mendukung bahwa kedua row adalah entity yang sama.
- Isi `review_label = 0` jika terdapat konflik identitas atau evidence hanya berasal dari field yang umum seperti DOB/city.
- Biarkan kosong jika bukti tidak cukup dan tambahkan alasan di `review_notes`.
- `customer_id` tidak ditampilkan dan tidak boleh dipakai sebagai jawaban otomatis.

In [ ]:
raw_identity_columns = [
    'first_name', 'last_name', 'email', 'phone_number',
    'dob', 'address', 'city', 'state', 'country',
]
available_identity_columns = [
    column for column in raw_identity_columns if column in df.columns
]

review_detail = review_queue[
    ['left_row_index', 'right_row_index', 'review_label', 'review_notes']
].copy()
indexed_df = df.set_index('row_index')
for side, index_column in [('left', 'left_row_index'), ('right', 'right_row_index')]:
    side_values = indexed_df.loc[
        review_detail[index_column].astype(int), available_identity_columns
    ].reset_index(drop=True)
    side_values.columns = [f'{side}_{column}' for column in available_identity_columns]
    review_detail = pd.concat([review_detail.reset_index(drop=True), side_values], axis=1)

detail_order = [
    'left_row_index', 'right_row_index', 'review_label', 'review_notes',
    *[f'left_{column}' for column in available_identity_columns],
    *[f'right_{column}' for column in available_identity_columns],
]
REVIEW_DETAIL_PATH = PROCESSED_DIR / 'manual_review_detail.csv'
review_detail = review_detail[detail_order]
if REVIEW_DETAIL_PATH.exists():
    existing_detail = pd.read_csv(REVIEW_DETAIL_PATH, sep=';')
    detail_key_columns = ['left_row_index', 'right_row_index']
    detail_update_columns = ['review_label', 'review_notes']
    existing_updates = existing_detail[
        detail_key_columns + detail_update_columns
    ].copy()
    existing_updates['review_label'] = pd.to_numeric(
        existing_updates['review_label'], errors='coerce'
    )
    review_detail = review_detail.drop(columns=detail_update_columns).merge(
        existing_updates,
        on=detail_key_columns,
        how='left',
        suffixes=('', '_existing'),
    )
    print('Loaded existing review detail without overwriting:', REVIEW_DETAIL_PATH)
else:
    review_detail.to_csv(REVIEW_DETAIL_PATH, index=False, sep=';')
    print('Saved review detail:', REVIEW_DETAIL_PATH)

review_key = ['left_row_index', 'right_row_index']
queue_indexed = review_queue.set_index(review_key)
detail_indexed = review_detail.set_index(review_key)
for column in ['review_label', 'review_notes']:
    detail_values = detail_indexed[column].reindex(queue_indexed.index)
    queue_indexed[column] = detail_values.combine_first(queue_indexed[column])
review_queue = queue_indexed.reset_index()
review_queue['review_label'] = pd.to_numeric(review_queue['review_label'], errors='coerce')
review_queue['review_status'] = review_queue['review_status'].where(
    review_queue['review_label'].isna(),
    'reviewed',
)
review_detail.head()

## Experiment 4 - Rule evaluation against reviewed labels

`review_label` bernilai `1` untuk same entity dan `0` untuk different entity. Rule prediction dibuat eksplisit dari `agreement_count`; rule ini bukan model probabilistic dan hanya dievaluasi pada reviewed sample.

In [ ]:
def classification_metrics(reviewed: pd.DataFrame, prediction_column: str) -> dict:
    labeled = reviewed.dropna(subset=['review_label', prediction_column]).copy()
    if labeled.empty:
        return {
            'labeled_rows': 0,
            'true_positive': pd.NA,
            'false_positive': pd.NA,
            'false_negative': pd.NA,
            'true_negative': pd.NA,
            'precision': 'N/A - ground truth unavailable',
            'recall': 'N/A - ground truth unavailable',
            'f1': 'N/A - ground truth unavailable',
        }

    actual = labeled['review_label'].astype(int)
    predicted = labeled[prediction_column].astype(int)
    true_positive = int(((actual == 1) & (predicted == 1)).sum())
    false_positive = int(((actual == 0) & (predicted == 1)).sum())
    false_negative = int(((actual == 1) & (predicted == 0)).sum())
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return {
        'labeled_rows': len(labeled),
        'true_positive': true_positive,
        'false_positive': false_positive,
        'false_negative': false_negative,
        'true_negative': int(((actual == 0) & (predicted == 0)).sum()),
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

reviewed = review_queue.dropna(subset=['review_label']).copy()
reviewed['prediction_agreement_ge_4'] = reviewed['agreement_count'].ge(4).astype(int)
reviewed['prediction_agreement_ge_2'] = reviewed['agreement_count'].ge(2).astype(int)

metric_rows = []
for rule_name, prediction_column in [
    ('agreement_count >= 4', 'prediction_agreement_ge_4'),
    ('agreement_count >= 2', 'prediction_agreement_ge_2'),
]:
    metric_rows.append({
        'rule': rule_name,
        **classification_metrics(reviewed, prediction_column),
    })

metric_summary = pd.DataFrame(metric_rows)
metric_summary.round(4)

## Experiment 5 - Persist evaluation artifacts

Artifact review dipisahkan dari raw dan comparison vector. Reviewer mengisi `review_label`, `review_status`, dan `review_notes` pada file review terpisah setelah memeriksa pasangan secara manual.

In [ ]:
OUTPUT_DIR = DATA_PATH.parent
INTEGRITY_PATH = OUTPUT_DIR / 'evaluation_integrity_summary.csv'
DIAGNOSTICS_PATH = OUTPUT_DIR / 'evaluation_diagnostics.csv'
METRIC_OUTPUT_PATH = OUTPUT_DIR / 'evaluation_metric_summary.csv'

if not queue_loaded_from_disk:
    review_queue.to_csv(REVIEW_QUEUE_PATH, index=False, sep=';')
    print('Saved new review queue:', REVIEW_QUEUE_PATH)
else:
    print('Preserved existing review queue:', REVIEW_QUEUE_PATH)
integrity_summary.to_csv(INTEGRITY_PATH, index=False)
coverage_diagnostics.to_csv(DIAGNOSTICS_PATH, index=False)
metric_summary.to_csv(METRIC_OUTPUT_PATH, index=False)

print('Saved:', INTEGRITY_PATH)
print('Saved:', DIAGNOSTICS_PATH)
print('Saved:', METRIC_OUTPUT_PATH)
print('Raw dataset still exists:', (DATA_PATH.parents[1] / 'raw' / 'crm_50000_customers_dirty_v3.csv').exists())

# Result, Analysis, and Decision

## Result aktual

Gunakan `integrity_summary`, `coverage_diagnostics`, `evidence_diagnostics`, `review_summary`, dan `metric_summary` sebagai sumber hasil aktual. Metrik hanya berlaku untuk reviewed sample dan rule yang tertulis eksplisit.

## Decision

Tahap evaluasi sudah menghasilkan metrik untuk dua rule agreement pada reviewed sample. Hasil ini belum otomatis mewakili seluruh candidate set atau seluruh dataset.

## Risk and limitations

- Sample manual review bukan random sample dari seluruh N x N pair; sample hanya berasal dari candidate set hasil blocking.
- Candidate di luar blocking tidak dapat dinilai pada tahap ini.
- Reference coverage deterministic bukan recall.
- Row index bukan identifier bisnis dan tidak boleh dipakai sebagai label entity.
- Agreement indicator bukan probabilitas.

## Next Experiment

Gunakan hasil reviewed sample untuk error analysis: terutama pair dengan agreement rendah tetapi label berbeda dari rule, serta batasan sample terstratifikasi. Setelah itu bandingkan deterministic, fuzzy, dan probabilistic baseline dengan validation set yang lebih luas.